In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms
import timm

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [48]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "tiny-vision-transformer",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [30]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [35]:

tiny_vit = timm.create_model("tiny_vit_11m_224", pretrained=True)

for param in tiny_vit.parameters():
    param.requires_grad = False

tiny_vit.reset_classifier(84)

gpu = torch.device("cuda")
tiny_vit = tiny_vit.to(gpu)


In [36]:
print(tiny_vit)

TinyVit(
  (patch_embed): PatchEmbed(
    (conv1): ConvNorm(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (act): GELU(approximate='none')
    (conv2): ConvNorm(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (stages): Sequential(
    (0): ConvLayer(
      (blocks): Sequential(
        (0): MBConv(
          (conv1): ConvNorm(
            (conv): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (act1): GELU(approximate='none')
          (conv2): ConvNorm(
            (conv): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=256, bias=Fals

In [45]:
for param in tiny_vit.stages[-1].parameters():
    param.requires_grad = True

In [47]:
for param in tiny_vit.parameters():
    param.requires_grad = True

In [37]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": tiny_vit.head.parameters(), "lr": 1e-3, "weight_decay": 1e-4},
    {"params": tiny_vit.stages.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [38]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [40]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(tiny_vit, val_dataloader)

        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [49]:
train_model(tiny_vit, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.7077755855446897, Validation Accuracy: 0.9054225224681827
Epoch: 2, Training Loss: 0.4993522290604139, Validation Accuracy: 0.9228542131612281
Epoch: 3, Training Loss: 0.4137668651073326, Validation Accuracy: 0.9294287954641414
Epoch: 4, Training Loss: 0.35851592460272375, Validation Accuracy: 0.9361240123047229
Epoch: 5, Training Loss: 0.3246389648377624, Validation Accuracy: 0.9369684540683998
Epoch: 6, Training Loss: 0.29481429134516474, Validation Accuracy: 0.941733518306291
Epoch: 7, Training Loss: 0.2738820902646859, Validation Accuracy: 0.9421557391881296
Epoch: 8, Training Loss: 0.25736374587305716, Validation Accuracy: 0.94438747813499
Epoch: 9, Training Loss: 0.24168516609914548, Validation Accuracy: 0.9461969962000121
Epoch: 10, Training Loss: 0.23115139418199093, Validation Accuracy: 0.9457747753181736
Epoch: 11, Training Loss: 0.2181108475583723, Validation Accuracy: 0.9479461969962
Epoch: 12, Training Loss: 0.20714058745079766, Validation Accura

In [50]:
accuracy = validate_model(tiny_vit, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 94.59881642584638
